# Observation 0 - Train-all vs train-segment by age threshold

Target scale is direct amount (`target_amount` or `Opportunity Amount USD`), not log-transformed.
This notebook computes and renders a single table with sample count, constrained mean/std, and R2/MAE comparisons.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.options.display.max_rows = 200
pd.options.display.max_columns = 200


In [2]:
base_dir = Path.cwd().resolve()
candidate_roots = [base_dir, base_dir.parent, base_dir.parent.parent]
input_path = None
project_root = None

for root in candidate_roots:
    candidate = root / "observation_0" / "observation0_full_with_maturity.csv"
    if candidate.exists():
        input_path = candidate
        project_root = root
        break

if input_path is None:
    raise FileNotFoundError("Could not locate observation0_full_with_maturity.csv")

out_csv = project_root / "observation_0" / "observation0_amount_method1_train_scope_vs_age_threshold.csv"
df = pd.read_csv(input_path)
print(f"Loaded rows: {len(df):,}")
print(f"Input: {input_path}")
print(f"Output csv: {out_csv}")


Loaded rows: 78,025
Input: /mnt/HC_Volume_104595655/home/goviedb/repos/mib/cars_analysis/observation_0/observation0_full_with_maturity.csv
Output csv: /mnt/HC_Volume_104595655/home/goviedb/repos/mib/cars_analysis/observation_0/observation0_amount_method1_train_scope_vs_age_threshold.csv


In [3]:
stable_base_features = [
    'Supplies Group',
    'Supplies Subgroup',
    'Region',
    'Route To Market',
    'Client Size By Revenue (USD)',
    'Client Size By Employee Count',
    'Revenue From Client Past Two Years (USD)',
    'Competitor Type',
]

df['supply_route_combo'] = (
    df['Supplies Group'].astype('string').fillna('Unknown') + '|' +
    df['Route To Market'].astype('string').fillna('Unknown')
)
df['region_route_combo'] = (
    df['Region'].astype('string').fillna('Unknown') + '|' +
    df['Route To Market'].astype('string').fillna('Unknown')
)
df['client_size_combo'] = (
    df['Client Size By Revenue (USD)'].astype('string').fillna('Unknown') + '|' +
    df['Client Size By Employee Count'].astype('string').fillna('Unknown')
)

creation_features = stable_base_features + [
    'supply_route_combo',
    'region_route_combo',
    'client_size_combo',
]

target_col = 'target_amount' if 'target_amount' in df.columns else 'Opportunity Amount USD'
work = df[creation_features + ['Elapsed Days In Sales Stage', target_col]].copy()
for ccol in creation_features:
    work[ccol] = work[ccol].astype('string').fillna('Unknown')
work['Elapsed Days In Sales Stage'] = pd.to_numeric(work['Elapsed Days In Sales Stage'], errors='coerce')
work[target_col] = pd.to_numeric(work[target_col], errors='coerce')
work = work.dropna(subset=['Elapsed Days In Sales Stage', target_col]).copy()
work['Elapsed Days In Sales Stage'] = work['Elapsed Days In Sales Stage'].astype(float)
work[target_col] = work[target_col].astype(float)
work = work.reset_index(drop=True)
work.shape


(78025, 13)

In [4]:
def make_model():
    pre = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), creation_features)],
        remainder='drop'
    )
    return Pipeline(steps=[
        ('pre', pre),
        ('ridge', Ridge(alpha=3.0)),
    ])

thresholds = [None, 10, 15, 20, 30, 45, 60, 75, 90, 120]
rows = []
all_idx = np.arange(len(work))

for t in thresholds:
    if t is None:
        seg_mask = np.ones(len(work), dtype=bool)
        label = 'all'
        min_age_exclusive = -1
    else:
        seg_mask = work['Elapsed Days In Sales Stage'].to_numpy() > float(t)
        label = f'> {t}'
        min_age_exclusive = int(t)

    seg_idx = np.where(seg_mask)[0]
    n = len(seg_idx)
    seg_y = work.loc[seg_idx, target_col] if n > 0 else pd.Series(dtype=float)
    seg_mean = float(seg_y.mean()) if n > 0 else np.nan
    seg_std = float(seg_y.std(ddof=1)) if n > 1 else np.nan

    if n < 200:
        rows.append({
            'threshold_label': label,
            'min_age_exclusive_days': min_age_exclusive,
            'n_rows_segment': n,
            'segment_mean_amount': seg_mean,
            'segment_std_amount': seg_std,
            'r2_train_all_eval_segment_mean': np.nan,
            'r2_train_all_eval_segment_std': np.nan,
            'r2_train_segment_eval_segment_mean': np.nan,
            'r2_train_segment_eval_segment_std': np.nan,
            'delta_r2_segment_minus_all': np.nan,
            'mae_train_all_eval_segment_mean': np.nan,
            'mae_train_segment_eval_segment_mean': np.nan,
            'delta_mae_all_minus_segment': np.nan,
        })
        continue

    n_splits = min(5, max(3, n // 100))
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_r2_all = []
    fold_r2_seg = []
    fold_mae_all = []
    fold_mae_seg = []

    for seg_train_pos, seg_test_pos in cv.split(seg_idx):
        test_idx = seg_idx[seg_test_pos]
        train_seg_idx = seg_idx[seg_train_pos]
        test_set = set(test_idx.tolist())
        train_all_idx = np.array([i for i in all_idx if i not in test_set], dtype=int)

        X_train_all = work.loc[train_all_idx, creation_features]
        y_train_all = work.loc[train_all_idx, target_col]
        X_train_seg = work.loc[train_seg_idx, creation_features]
        y_train_seg = work.loc[train_seg_idx, target_col]
        X_test = work.loc[test_idx, creation_features]
        y_test = work.loc[test_idx, target_col]

        model_all = make_model()
        model_seg = make_model()
        model_all.fit(X_train_all, y_train_all)
        model_seg.fit(X_train_seg, y_train_seg)

        pred_all = model_all.predict(X_test)
        pred_seg = model_seg.predict(X_test)

        fold_r2_all.append(r2_score(y_test, pred_all))
        fold_r2_seg.append(r2_score(y_test, pred_seg))
        fold_mae_all.append(mean_absolute_error(y_test, pred_all))
        fold_mae_seg.append(mean_absolute_error(y_test, pred_seg))

    rows.append({
        'threshold_label': label,
        'min_age_exclusive_days': min_age_exclusive,
        'n_rows_segment': n,
        'segment_mean_amount': seg_mean,
        'segment_std_amount': seg_std,
        'r2_train_all_eval_segment_mean': float(np.mean(fold_r2_all)),
        'r2_train_all_eval_segment_std': float(np.std(fold_r2_all)),
        'r2_train_segment_eval_segment_mean': float(np.mean(fold_r2_seg)),
        'r2_train_segment_eval_segment_std': float(np.std(fold_r2_seg)),
        'delta_r2_segment_minus_all': float(np.mean(fold_r2_seg) - np.mean(fold_r2_all)),
        'mae_train_all_eval_segment_mean': float(np.mean(fold_mae_all)),
        'mae_train_segment_eval_segment_mean': float(np.mean(fold_mae_seg)),
        'delta_mae_all_minus_segment': float(np.mean(fold_mae_all) - np.mean(fold_mae_seg)),
    })

res = pd.DataFrame(rows).sort_values('min_age_exclusive_days').reset_index(drop=True)
res.to_csv(out_csv, index=False)
res


,threshold_label,min_age_exclusive_days,n_rows_segment,segment_mean_amount,segment_std_amount,r2_train_all_eval_segment_mean,r2_train_all_eval_segment_std,r2_train_segment_eval_segment_mean,r2_train_segment_eval_segment_std,delta_r2_segment_minus_all,mae_train_all_eval_segment_mean,mae_train_segment_eval_segment_mean,delta_mae_all_minus_segment
0,all,-1,78025,91637.260750,133161.029156,0.119258,0.004657,0.119258,0.004657,0.000000,74511.308081,74511.308081,0.000000
1,> 10,10,69533,91332.492514,132892.467678,0.120327,0.004814,0.120335,0.004828,0.000008,74525.397495,74219.049772,306.347723
2,> 15,15,66273,91227.728819,132661.899488,0.122018,0.001718,0.122315,0.001681,0.000297,74235.480093,73848.998907,386.481187
3,> 20,20,57679,91164.165589,132739.000702,0.123592,0.005658,0.125207,0.005225,0.001615,74451.990127,73527.149074,924.841053
4,> 30,30,47903,90793.216166,131908.279456,0.123087,0.003606,0.125287,0.003939,0.002200,74695.440057,73132.863090,1562.576967
5,> 45,45,34885,90215.160327,130731.549261,0.128153,0.013621,0.131937,0.010186,0.003784,74478.904998,72206.563389,2272.341609
6,> 60,60,24071,88322.225209,127919.224425,0.131721,0.006648,0.138725,0.006659,0.007003,73757.070214,70395.497966,3361.572248
7,> 75,75,12327,86532.376085,125544.831420,0.117772,0.009225,0.131367,0.006295,0.013595,74771.880003,69849.573340,4922.306663
8,> 90,90,1678,87975.921335,144391.954226,0.108175,0.032449,0.073881,0.048301,-0.034293,80145.774844,82659.170150,-2513.395305
9,> 120,120,38,82099.315789,120337.640257,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
table = res[[
    'threshold_label',
    'n_rows_segment',
    'segment_mean_amount',
    'segment_std_amount',
    'r2_train_all_eval_segment_mean',
    'r2_train_segment_eval_segment_mean',
    'delta_r2_segment_minus_all',
]].copy()
table


,threshold_label,n_rows_segment,segment_mean_amount,segment_std_amount,r2_train_all_eval_segment_mean,r2_train_segment_eval_segment_mean,delta_r2_segment_minus_all
0,all,78025,91637.260750,133161.029156,0.119258,0.119258,0.000000
1,> 10,69533,91332.492514,132892.467678,0.120327,0.120335,0.000008
2,> 15,66273,91227.728819,132661.899488,0.122018,0.122315,0.000297
3,> 20,57679,91164.165589,132739.000702,0.123592,0.125207,0.001615
4,> 30,47903,90793.216166,131908.279456,0.123087,0.125287,0.002200
5,> 45,34885,90215.160327,130731.549261,0.128153,0.131937,0.003784
6,> 60,24071,88322.225209,127919.224425,0.131721,0.138725,0.007003
7,> 75,12327,86532.376085,125544.831420,0.117772,0.131367,0.013595
8,> 90,1678,87975.921335,144391.954226,0.108175,0.073881,-0.034293
9,> 120,38,82099.315789,120337.640257,NaN,NaN,NaN
